In [4]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import polars as pl

In [5]:
# load cpi embeddings
parquet_path = "/home/jko/ssl-cpi-analysis/data/SSL-Model-v3/cpi3m_campaign_cls_head_features_compressed.parquet"
df = pl.read_parquet(parquet_path)
unique_campaigns = df["campaign"].unique()
print("Unique campaigns:", unique_campaigns.to_list())

Unique campaigns: ['IPHEX', 'CRYSTAL_FACE_NASA', 'MIDCIX', 'ATTREX', 'MPACE', 'MC3E', 'AIRS_II', 'ARM', 'ISDAC', 'ICE_L', 'MACPEX', 'CRYSTAL_FACE_UND']


In [8]:
counts = (
    df
    .with_columns(pl.col("campaign").str.replace_all("-", "_"))
    .group_by("campaign")
    .len()
    .sort("campaign")
)
print(counts.to_pandas())
print(counts.shape)

             campaign      len
0             AIRS_II    92201
1                 ARM   295703
2              ATTREX   129128
3   CRYSTAL_FACE_NASA    78152
4    CRYSTAL_FACE_UND  1617826
5               ICE_L    46236
6               IPHEX    40692
7               ISDAC   505812
8              MACPEX    80240
9                MC3E   187558
10             MIDCIX    90761
11              MPACE    36042
(12, 2)


In [10]:
df.head()

campaign_file_id,cls_features,campaign,filename,head_features
str,list[f32],str,str,list[f32]
"""AIRS_II/1114-115708_753_18.png""","[-1.573681, -1.34678, … 1.054788]","""AIRS_II""","""1114-115708_753_18.png""","[-0.174372, 0.220068, … 0.195601]"
"""AIRS_II/1114-115708_753_25.png""","[-0.664088, -0.620979, … 1.610363]","""AIRS_II""","""1114-115708_753_25.png""","[0.150885, 0.086997, … 0.107423]"
"""AIRS_II/1114-115708_753_35.png""","[1.181937, -2.416504, … 0.973358]","""AIRS_II""","""1114-115708_753_35.png""","[0.008603, 0.284243, … 0.01234]"
"""AIRS_II/1114-115708_753_4.png""","[1.407759, -2.779922, … 0.635076]","""AIRS_II""","""1114-115708_753_4.png""","[-0.098592, 0.049973, … 0.028704]"
"""AIRS_II/1114-115745_814_121.pn…","[0.783167, -2.249387, … -0.949386]","""AIRS_II""","""1114-115745_814_121.png""","[-0.042272, -0.122378, … -0.023314]"


In [9]:
# Extract datetime from filename once
df_with_dt = df.with_columns(
    pl.col("filename")
    .str.extract(r"(\d{4}_\d{4}_\d{6})")
    .str.strptime(pl.Datetime, format="%Y_%m%d_%H%M%S")
    .alias("datetime")
)

# Get date range for each campaign
date_ranges = (
    df_with_dt
    .group_by("campaign")
    .agg([
        pl.col("datetime").min().alias("start_date"),
        pl.col("datetime").max().alias("end_date"),
        pl.len().alias("n_samples")
    ])
    .sort("campaign")
)

# Print all rows
pl.Config.set_tbl_rows(-1)

print(date_ranges)

InvalidOperationError: conversion from `str` to `datetime[μs]` failed in column 'filename' for 1 out of 100010 values: ["2004_0931_010828"]

You might want to try:
- setting `strict=False` to set values that cannot be converted to `null`
- using `str.strptime`, `str.to_date`, or `str.to_datetime` and providing a format string

In [11]:
# return rows where filename starts with "2004_0931_010828"
bad_rows = df.filter(
    pl.col("filename").str.starts_with("2004_0931_010828")
)

print(bad_rows)

shape: (1, 5)
┌───────────────────────┬───────────────────────┬──────────┬───────────────────────┬───────────────┐
│ campaign_file_id      ┆ cls_features          ┆ campaign ┆ filename              ┆ head_features │
│ ---                   ┆ ---                   ┆ ---      ┆ ---                   ┆ ---           │
│ str                   ┆ list[f32]             ┆ str      ┆ str                   ┆ list[f32]     │
╞═══════════════════════╪═══════════════════════╪══════════╪═══════════════════════╪═══════════════╡
│ MPACE/2004_0931_01082 ┆ [-0.744011, 1.889707, ┆ MPACE    ┆ 2004_0931_010828_132_ ┆ [0.019731,    │
│ 8_132_10.…            ┆ … 1.9517…             ┆          ┆ 10.png                ┆ 0.013027, …   │
│                       ┆                       ┆          ┆                       ┆ 0.03092…      │
└───────────────────────┴───────────────────────┴──────────┴───────────────────────┴───────────────┘


In [18]:
# Subset MPACE campaign first
mpace_df = df.filter(
    pl.col("campaign") == "MPACE"
)

# Convert filename timestamp -> datetime
mpace_df = mpace_df.with_columns(
    pl.col("filename")
    .str.extract(r"(\d{4}_\d{4}_\d{6})")
    .str.strptime(
        pl.Datetime,
        format="%Y_%m%d_%H%M%S",
        strict=False
    )
    .alias("datetime")
)

# Get date range
min_date = mpace_df["datetime"].min()
max_date = mpace_df["datetime"].max()

print(f"Min date: {min_date}")
print(f"Max date: {max_date}")

Min date: 2004-09-30 01:01:20
Max date: 2004-10-22 00:51:01


In [21]:
import polars as pl

# Subset MPACE and parse datetime
mpace_df = (
    df
    .filter(pl.col("campaign") == "MPACE")
    .with_columns(
        pl.col("filename")
        .str.extract(r"(\d{4}_\d{4}_\d{6})")
        .str.strptime(
            pl.Datetime,
            format="%Y_%m%d_%H%M%S",
            strict=False
        )
        .alias("datetime")
    )
)

# Count samples per calendar date
daily_counts = (
    mpace_df
    .with_columns(
        pl.col("datetime").dt.date().alias("date")
    )
    .group_by("date")
    .len()
    .sort("date")
)

pl.Config.set_tbl_rows(-1)

print(daily_counts)

shape: (15, 2)
┌────────────┬───────┐
│ date       ┆ len   │
│ ---        ┆ ---   │
│ date       ┆ u32   │
╞════════════╪═══════╡
│ null       ┆ 1     │
│ 2004-09-30 ┆ 35    │
│ 2004-10-05 ┆ 6882  │
│ 2004-10-06 ┆ 6630  │
│ 2004-10-07 ┆ 4     │
│ 2004-10-08 ┆ 603   │
│ 2004-10-09 ┆ 474   │
│ 2004-10-10 ┆ 1790  │
│ 2004-10-12 ┆ 258   │
│ 2004-10-13 ┆ 182   │
│ 2004-10-17 ┆ 13564 │
│ 2004-10-18 ┆ 1888  │
│ 2004-10-20 ┆ 533   │
│ 2004-10-21 ┆ 1707  │
│ 2004-10-22 ┆ 1491  │
└────────────┴───────┘
